In [1]:
%matplotlib inline
import os
import sys
from pathlib import Path
from urllib.parse import quote_plus

import pandas as pd

from dotenv import load_dotenv
from sqlalchemy import create_engine


_NOTEBOOK_DIR = Path.cwd()
_PROJECT_ROOT = _NOTEBOOK_DIR
for _ in range(5):
    if (_PROJECT_ROOT / ".env").exists():
        break
    _PROJECT_ROOT = _PROJECT_ROOT.parent

def _load_env():
    load_dotenv(_PROJECT_ROOT / ".env")

def _db_url():
    host = os.getenv("PGHOST") or os.getenv("DB_HOST", "localhost")
    port = os.getenv("PGPORT") or os.getenv("DB_PORT", "5432")
    user = os.getenv("PGUSER") or os.getenv("DB_USER", "postgres")
    password = os.getenv("PGPASSWORD") or os.getenv("DB_PASS", "")
    dbname = os.getenv("PGDATABASE") or os.getenv("DB_NAME", "baseball")
    pw = quote_plus(password) if password else ""
    return f"postgresql://{user}:{pw}@{host}:{port}/{dbname}"

_load_env()
engine = create_engine(_db_url())

In [2]:
from sklearn.metrics import log_loss, accuracy_score, f1_score
from sklearn.utils.class_weight import compute_class_weight

import numpy as np
import optuna
import xgboost as xgb

sys.path.insert(0, str(_NOTEBOOK_DIR))
from data_prep_batters import (
    load_and_prepare_batter,
    PITCH_RESULT_CLASSES,
    RESULT_TO_IDX,
    IDX_TO_RESULT,
    BUCKET_CLASS_NAMES,
    NUM_BUCKET_CLASSES,
)

print("Imports OK.")

Imports OK.


c:\Users\macia\AppData\Local\Programs\Python\Python313\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
X_train, y_train, X_val, y_val, X_test, y_test, cat_encodings, feature_cols = load_and_prepare_batter(
    engine, time_based=False, train_frac=0.8, val_frac=0.1, random_state=42
)

# 5-bucket target (at-bat outcome) or 10-class pitch_result
use_buckets = np.issubdtype(y_train.dtype, np.integer) and set(np.unique(y_train)).issubset({0, 1, 2, 3, 4})
if use_buckets:
    CLASS_NAMES = BUCKET_CLASS_NAMES
    num_class = NUM_BUCKET_CLASSES
    y_train_idx = y_train.astype(int)
    y_val_idx = y_val.astype(int)
    y_test_idx = y_test.astype(int)
else:
    CLASS_NAMES = PITCH_RESULT_CLASSES
    num_class = len(PITCH_RESULT_CLASSES)
    y_train_idx = y_train.map(lambda r: RESULT_TO_IDX.get(r, 0)).astype(int)
    y_val_idx = y_val.map(lambda r: RESULT_TO_IDX.get(r, 0)).astype(int)
    y_test_idx = y_test.map(lambda r: RESULT_TO_IDX.get(r, 0)).astype(int)

print(f"Train: {len(X_train)} | Val: {len(X_val)} | Test: {len(X_test)}")
print(f"Features: {feature_cols}")
print(f"Target: {'5-bucket (at-bat outcome)' if use_buckets else '10-class pitch_result'}, n_classes={num_class}")
print("\nTarget distribution (train):")
print(y_train.value_counts())

Train: 288673 | Val: 36084 | Test: 36085
Features: ['pitch_type', 'plate_x', 'plate_z', 'release_speed', 'release_spin_rate', 'balls', 'strikes', 'stand', 'p_throws', 'batter', 'career_AB', 'career_H', 'career_HR', 'career_BA', 'career_OBP', 'career_SLG', 'home_team', 'inning', 'outs_when_up', 'is_pitcher_count', 'is_batter_count', 'pitcher', 'previous_pitch_type', 'previous_was_strike', 'in_zone', 'is_fastball', 'game_year']
Target: 5-bucket (at-bat outcome), n_classes=5

Target distribution (train):
target_class
0    112342
1     82155
2     37921
3     36769
4     19486
Name: count, dtype: int64


In [4]:
# Train on full data; use targeted upweight for weak outcomes only (modest bump to avoid tanking accuracy).
X_train_bal = X_train
y_train_idx_bal = y_train_idx
if num_class == 5:
    # 5-bucket: weak = single (3), extra_base (4)
    WEAK_CLASS_NAMES = ["single", "extra_base"]
    weak_class_indices = np.array([CLASS_NAMES.index(c) for c in WEAK_CLASS_NAMES])
else:
    WEAK_CLASS_NAMES = ["swinging_strike", "in_play_out", "in_play_1b", "in_play_2b", "in_play_3b", "in_play_hr"]
    weak_class_indices = np.array([CLASS_NAMES.index(c) for c in WEAK_CLASS_NAMES])
TARGETED_WEIGHT_WEAK = 1.5
class_weights = np.ones(num_class)
class_weights[weak_class_indices] = TARGETED_WEIGHT_WEAK
sample_weight_batter = class_weights[y_train_idx_bal.to_numpy().ravel()]
print(f"Train: n={len(X_train)} with targeted weights (weak classes {TARGETED_WEIGHT_WEAK}x)")

Train: n=288673 with targeted weights (weak classes 1.5x)


In [5]:
# Hyperparameter tuning: Optuna minimizes blend of (1 - macro F1) and log loss so
# the model cares about both rare-class recall and calibration; 30 trials.
def tune_batter_pitch_result(trial):
    n_estimators = trial.suggest_int("n_estimators", 50, 500)
    max_depth = trial.suggest_int("max_depth", 3, 12)
    learning_rate = trial.suggest_float("learning_rate", 0.01, 0.3)
    subsample = trial.suggest_float("subsample", 0.5, 1.0)
    colsample_bytree = trial.suggest_float("colsample_bytree", 0.5, 1.0)
    reg_alpha = trial.suggest_float("reg_alpha", 0.1, 10.0)
    reg_lambda = trial.suggest_float("reg_lambda", 0.1, 10.0)
    min_child_weight = trial.suggest_int("min_child_weight", 1, 10)
    model = xgb.XGBClassifier(
        objective="multi:softprob",
        num_class=num_class,
        random_state=42,
        n_estimators=n_estimators,
        max_depth=max_depth,
        learning_rate=learning_rate,
        subsample=subsample,
        colsample_bytree=colsample_bytree,
        reg_alpha=reg_alpha,
        reg_lambda=reg_lambda,
        min_child_weight=min_child_weight,
    )
    model.fit(
        X_train_bal[feature_cols],
        y_train_idx_bal,
        sample_weight=sample_weight_batter,
    )
    proba = model.predict_proba(X_val[feature_cols])
    pred = np.argmax(proba, axis=1)
    macro_f1 = f1_score(y_val_idx, pred, average="macro", zero_division=0)
    logls = log_loss(y_val_idx, proba, labels=np.arange(num_class))
    return 0.6 * (1.0 - macro_f1) + 0.4 * logls

study_batter = optuna.create_study(direction="minimize")
study_batter.optimize(tune_batter_pitch_result, n_trials=50)
print("Best blended (val):", study_batter.best_value, "| Best params:", study_batter.best_params)

[I 2026-02-26 21:19:27,155] A new study created in memory with name: no-name-4dff405d-cbe7-4cef-ba68-b1c356a9ed54
[I 2026-02-26 21:19:42,957] Trial 0 finished with value: 0.8248363435867464 and parameters: {'n_estimators': 343, 'max_depth': 7, 'learning_rate': 0.2877232980217471, 'subsample': 0.7147393277608591, 'colsample_bytree': 0.8514241083871308, 'reg_alpha': 0.11665147991771954, 'reg_lambda': 2.0035139905877477, 'min_child_weight': 5}. Best is trial 0 with value: 0.8248363435867464.
[I 2026-02-26 21:19:50,901] Trial 1 finished with value: 0.8639602200420171 and parameters: {'n_estimators': 86, 'max_depth': 12, 'learning_rate': 0.17515460435229557, 'subsample': 0.8917473623996244, 'colsample_bytree': 0.9622930883279122, 'reg_alpha': 5.547409134873142, 'reg_lambda': 7.619095102045911, 'min_child_weight': 2}. Best is trial 0 with value: 0.8248363435867464.
[I 2026-02-26 21:19:59,826] Trial 2 finished with value: 0.9622938210701625 and parameters: {'n_estimators': 343, 'max_depth': 4

Best blended (val): 0.7149877217534425 | Best params: {'n_estimators': 466, 'max_depth': 11, 'learning_rate': 0.192427568658527, 'subsample': 0.937736925613406, 'colsample_bytree': 0.9682039312673614, 'reg_alpha': 0.7111805882879177, 'reg_lambda': 5.900337181644298, 'min_child_weight': 2}


In [6]:
# XGBoost multiclass classifier for pitch_result (hyperparameters from Optuna)
clf = xgb.XGBClassifier(
    objective="multi:softprob",
    num_class=num_class,
    random_state=42,
    **study_batter.best_params,
)
clf.fit(
    X_train_bal[feature_cols],
    y_train_idx_bal,
    sample_weight=sample_weight_batter,
    eval_set=[(X_val[feature_cols], y_val_idx)],
    verbose=20,
)

[0]	validation_0-mlogloss:1.41840
[20]	validation_0-mlogloss:1.31272
[40]	validation_0-mlogloss:1.27199
[60]	validation_0-mlogloss:1.23527
[80]	validation_0-mlogloss:1.20053
[100]	validation_0-mlogloss:1.17374
[120]	validation_0-mlogloss:1.15258
[140]	validation_0-mlogloss:1.13541
[160]	validation_0-mlogloss:1.11856
[180]	validation_0-mlogloss:1.10501
[200]	validation_0-mlogloss:1.09314
[220]	validation_0-mlogloss:1.08384
[240]	validation_0-mlogloss:1.07554
[260]	validation_0-mlogloss:1.06604
[280]	validation_0-mlogloss:1.06154
[300]	validation_0-mlogloss:1.05477
[320]	validation_0-mlogloss:1.05068
[340]	validation_0-mlogloss:1.04615
[360]	validation_0-mlogloss:1.04259
[380]	validation_0-mlogloss:1.03888
[400]	validation_0-mlogloss:1.03543
[420]	validation_0-mlogloss:1.03250
[440]	validation_0-mlogloss:1.03030
[460]	validation_0-mlogloss:1.02893
[465]	validation_0-mlogloss:1.02876


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.9682039312673614, device=None,
              early_stopping_rounds=None, enable_categorical=False,
              eval_metric=None, feature_types=None, feature_weights=None,
              gamma=None, grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.192427568658527,
              max_bin=None, max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=11, max_leaves=None,
              min_child_weight=2, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=466, n_jobs=None, num_class=5, ...)

In [ ]:
# (Threshold tuning runs in the calibration cell below, after clf_calibrated is fitted.)

In [7]:
# Calibrate probabilities (isotonic on validation set) for run expectancy / sampling
from sklearn.calibration import CalibratedClassifierCV

clf_calibrated = CalibratedClassifierCV(clf, method="isotonic", cv="prefit")
clf_calibrated.fit(X_val[feature_cols], y_val_idx)
print("Calibrated classifier fitted on validation set (use for predict_proba)")

# Tune thresholds for weak classes: extended grid (lower T) then per-class thresholds.
def predict_with_weak_threshold(proba, threshold, weak_indices):
    pred = np.argmax(proba, axis=1)
    weak_proba = proba[:, weak_indices]
    weak_argmax_idx = np.argmax(weak_proba, axis=1)
    weak_max = np.max(weak_proba, axis=1)
    for i in range(len(pred)):
        if weak_max[i] >= threshold and weak_max[i] >= proba[i, pred[i]]:
            pred[i] = weak_indices[weak_argmax_idx[i]]
    return pred

def predict_with_per_class_threshold(proba, threshold_per_class, weak_indices):
    """threshold_per_class: list of length len(weak_indices), one T per weak class."""
    pred = np.argmax(proba, axis=1)
    weak_indices = np.asarray(weak_indices)
    threshold_per_class = np.asarray(threshold_per_class)
    for i in range(len(pred)):
        best_weak_k, best_weak_p = -1, -1.0
        for j, k in enumerate(weak_indices):
            if proba[i, k] >= threshold_per_class[j] and proba[i, k] > best_weak_p:
                best_weak_p = proba[i, k]
                best_weak_k = k
        if best_weak_k >= 0 and best_weak_p >= proba[i, pred[i]]:
            pred[i] = best_weak_k
    return pred

val_proba = clf_calibrated.predict_proba(X_val[feature_cols])
# 1) Single global T with extended grid (include 0.02, 0.03, 0.04 for rarer classes)
grid_single = [0.02, 0.03, 0.04, 0.05, 0.08, 0.10, 0.12, 0.15, 0.20]
best_f1, threshold_weak = 0.0, 0.10
for T in grid_single:
    y_val_pred = predict_with_weak_threshold(val_proba, T, weak_class_indices)
    f1 = f1_score(y_val_idx, y_val_pred, average="macro", zero_division=0)
    if f1 > best_f1:
        best_f1, threshold_weak = f1, T
print(f"Single threshold (val): best macro F1 = {best_f1:.4f} at threshold_weak = {threshold_weak}")

# 2) Per-class thresholds: iterative tuning (one class at a time)
grid_per_class = [0.02, 0.04, 0.06, 0.08, 0.10, 0.12]
threshold_per_weak_class = [threshold_weak] * len(weak_class_indices)
for j in range(len(weak_class_indices)):
    best_f1_j, best_t_j = 0.0, threshold_per_weak_class[j]
    for T in grid_per_class:
        threshold_per_weak_class[j] = T
        y_val_pred = predict_with_per_class_threshold(val_proba, threshold_per_weak_class, weak_class_indices)
        f1 = f1_score(y_val_idx, y_val_pred, average="macro", zero_division=0)
        if f1 > best_f1_j:
            best_f1_j, best_t_j = f1, T
    threshold_per_weak_class[j] = best_t_j
y_val_final = predict_with_per_class_threshold(val_proba, threshold_per_weak_class, weak_class_indices)
f1_per_class = f1_score(y_val_idx, y_val_final, average="macro", zero_division=0)
print(f"Per-class thresholds (val): macro F1 = {f1_per_class:.4f}")
print("Per-class T:", dict(zip(WEAK_CLASS_NAMES, threshold_per_weak_class)))

c:\Users\macia\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\calibration.py:333: UserWarning: The `cv='prefit'` option is deprecated in 1.6 and will be removed in 1.8. You can use CalibratedClassifierCV(FrozenEstimator(estimator)) instead.
  warnings.warn(


Calibrated classifier fitted on validation set (use for predict_proba)
Single threshold (val): best macro F1 = 0.5314 at threshold_weak = 0.02
Per-class thresholds (val): macro F1 = 0.5314
Per-class T: {'single': 0.02, 'extra_base': 0.02}


In [8]:
import json
import joblib
import shutil

_SAVED = Path(_NOTEBOOK_DIR) / "saved_models"
_SAVED.mkdir(parents=True, exist_ok=True)

clf.save_model(str(_SAVED / "batter_pitch_result.json"))
print("Saved batter_pitch_result.json")

# Calibrated model for probability outputs (run expectancy / sampling)
joblib.dump(clf_calibrated, _SAVED / "batter_pitch_result_calibrated.joblib")
print("Saved batter_pitch_result_calibrated.joblib")

# Encoders for inference (same column order and mappings); per-class thresholds for weak-class recall
_result_to_idx = {CLASS_NAMES[i]: i for i in range(num_class)}
_idx_to_result = {str(i): CLASS_NAMES[i] for i in range(num_class)}
batter_encoders = {
    "feats_batter": feature_cols,
    "result_to_idx": _result_to_idx,
    "idx_to_result": _idx_to_result,
    "cat_encodings": cat_encodings,
    "pitch_result_classes": list(CLASS_NAMES),
    "threshold_weak": float(np.mean(threshold_per_weak_class)),
    "weak_class_indices": weak_class_indices.tolist(),
    "threshold_per_weak_class": [float(t) for t in threshold_per_weak_class],
}
with open(_SAVED / "batter_encoders.json", "w") as f:
    json.dump(batter_encoders, f, indent=2)
print("Saved batter_encoders.json")

# Copy to Models/saved_models so web app and evaluation can load
_MODELS = _PROJECT_ROOT / "Models" / "saved_models"
_MODELS.mkdir(parents=True, exist_ok=True)
shutil.copy(_SAVED / "batter_pitch_result.json", _MODELS / "batter_pitch_result.json")
shutil.copy(_SAVED / "batter_encoders.json", _MODELS / "batter_encoders.json")
if (_SAVED / "batter_pitch_result_calibrated.joblib").exists():
    shutil.copy(_SAVED / "batter_pitch_result_calibrated.joblib", _MODELS / "batter_pitch_result_calibrated.joblib")
print("Copied to Models/saved_models")

Saved batter_pitch_result.json
Saved batter_pitch_result_calibrated.joblib
Saved batter_encoders.json
Copied to Models/saved_models


In [9]:
# Phase 4.1: Overall accuracy and log-loss on test set (using calibrated model)
y_pred = clf_calibrated.predict(X_test[feature_cols])
y_proba = clf_calibrated.predict_proba(X_test[feature_cols])
acc = accuracy_score(y_test_idx, y_pred)
ll = log_loss(y_test_idx, y_proba, labels=np.arange(num_class))
print(f"Test accuracy: {acc:.4f}")
print(f"Test log-loss: {ll:.4f}")

Test accuracy: 0.6020
Test log-loss: 1.0198


In [10]:
# Phase 4.2: Per-batter metrics (batters with >= 50 test pitches)
if "batter" in X_test.columns:
    test_df = X_test[feature_cols].copy()
    test_df["y_true"] = y_test_idx.values
    test_df["y_pred"] = y_pred
    def _acc(g):
        return accuracy_score(g["y_true"], g["y_pred"])
    per_batter = test_df.groupby("batter").agg(n=("y_true", "count")).reset_index()
    per_batter = per_batter[per_batter["n"] >= 50]
    per_batter["acc"] = per_batter["batter"].apply(
        lambda b: accuracy_score(test_df.loc[test_df["batter"] == b, "y_true"], test_df.loc[test_df["batter"] == b, "y_pred"])
    )
    per_batter = per_batter.sort_values("n", ascending=False)
    print("Per-batter accuracy (min 50 test pitches, top 10 by volume):")
    print(per_batter.head(10))

Per-batter accuracy (min 50 test pitches, top 10 by volume):
     batter    n       acc
101  621566  172  0.575581
156  646240  170  0.582353
383  672695  169  0.639053
62   596019  163  0.533742
183  656941  156  0.557692
599  695578  155  0.593548
107  624413  154  0.551948
251  665742  151  0.503311
564  691406  150  0.553333
142  642715  149  0.664430
